In [9]:
import pandas as pd
from Bio import SeqIO  
import numpy as np
import os
import multiprocessing as mp
from ete3 import NCBITaxa

## get clustered NR database

In [364]:
!sbatch scripts/get_clustered_nr.sh

## get B. coprophila genome annotation and extract longest isoform 

In [1]:
%%bash
## downlaod B. coprophila GCF_014529535.2-RS_2026_02 5/26/2026
curl -L -o ncbi_dataset.zip \
"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/genome/accession/GCF_014529535.2/download?include_annotation_type=GENOME_GFF&include_annotation_type=GENOME_FASTA&include_annotation_type=PROT_FASTA&hydrated=FULLY_HYDRATED"

unzip -o ncbi_dataset.zip
rm ncbi_dataset.zip

In [10]:

import pandas as pd
gff=pd.read_csv("ncbi_dataset/data/GCF_014529535.2/genomic.gff",sep="\t",comment="#",header=None)
def get_gene(x):
    return gff[gff[8].str.contains(x)].iloc[0,8].split(";gene=")[1].split(";")[0]
    
input_file = open(f"ncbi_dataset/data/GCF_014529535.2/protein.faa") 
fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))

In [11]:
cds=gff[gff[2]=='CDS']
cds['prot']=[x.split(";")[0].split("-")[1] for x in cds[8]]
cds['gene']=[x.split(";gene=")[1].split(";")[0] for x in cds[8]]

/tmp/ipykernel_1123499/3762869742.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cds['prot']=[x.split(";")[0].split("-")[1] for x in cds[8]]
/tmp/ipykernel_1123499/3762869742.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cds['gene']=[x.split(";gene=")[1].split(";")[0] for x in cds[8]]


In [12]:
from collections import defaultdict
cds_gene={}
gene_cds=defaultdict(set)
for index, row in cds.iterrows():
    gene_cds[row.gene].add(row.prot)
    cds_gene[row.prot]=row.gene

In [24]:
cds_len={x:len(str(fasta[x].seq)) for x in fasta}
longest = []
for gene, proteins in gene_cds.items():
    max_len=max([cds_len[x] for x in proteins])
    longest.append([x for x in proteins if cds_len[x]==max_len][0])
len(longest)

16462

In [4]:
!sbatch scripts/update_taxonomy.sh 

Submitted batch job 17383896


In [30]:
longest_fasta={x:fasta[x] for x in longest}

In [31]:

SeqIO.write(
    longest_fasta.values(),
    "ncbi_dataset/data/GCF_014529535.2/longest_isoform.fasta",
    "fasta"
)

16462

In [40]:
!sbatch scripts/diamond_clustered.sh

Submitted batch job 16989147


In [48]:
!sh scripts/split_diamond.sh "$SCRATCH/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split" "$SCRATCH/extavour_lab/Everyone/Rishabh/diamond_clustered_results" 


Splitting complete. Split files are stored in /n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split directory.


In [55]:
from ete3 import NCBITaxa
import numpy as np
ncbi = NCBITaxa()
def taxid_to_lineage_string(taxid):
    try: 
        lineage = ncbi.get_lineage(taxid)
        names = ncbi.get_taxid_translator(lineage)
        
        ranks = ncbi.get_rank(lineage)
    
        
        kingdom = np.nan
        phylum = np.nan
        out={}
    
        for tid in lineage:
            rank = ranks[tid]
            if rank == 'kingdom':
                kingdom=names[tid]
            elif rank =='phylum':
                phylum=names[tid]
        names = "; ".join(names[t] for t in lineage).split("root; ")[1]
    except:
        print("fail",taxid)
        names=np.nan     
        kingdom = np.nan
        phylum = np.nan
        
    try:
       name=ncbi.get_taxid_translator([taxid])[taxid] 
    except:
        name=np.nan
  
    return names, name, kingdom, phylum



In [60]:
!mkdir "/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split_edited" "/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustred_dipteran_results_split/"

In [61]:
import os
import pandas as pd
import multiprocessing as mp

def get_missing_tax_add_header(x):
    df = pd.read_csv(
        f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split/{x}",
        sep="\t",
        names="qseqid sseqid stitle staxids sscinames sphylums skingdoms slineage pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovhsp".split(),
        nrows=2000
    )

    tids = set(df.loc[df.slineage.isna(), "staxids"].dropna())

    df.to_csv(
        f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split_edited/{x}",
        sep="\t",
        index=False
    )

    return tids


In [62]:
import os
td=os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split")
td=[x for x in td if '.tsv' in x]
import multiprocessing as mp
num_processes = 20  # Use all available CPU cores
pool = mp.Pool(processes=num_processes)

# Use the pool's map function to apply the get_data function to each element in td
result = pool.map(get_missing_tax_add_header, td)

In [64]:
missing_tax = set().union(*result)
tid_map={x:taxid_to_lineage_string(x) for x in missing_tax}


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 3015169 was translated into 2991499
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 2723080 was translated into 2723081
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 68043 was translated into 1920
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 1917965 was translated into 1872424
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/n/home11/rkapoor/.conda/envs

In [69]:
def fill_missing_taxa(x):
    df = pd.read_csv(
        f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split_edited/{x}",
        sep="\t",
        nrows=2000,
        
    )
    for index, row in df.iterrows():
        if row.staxids in tid_map:
            df.loc[index,['slineage', 'sscinames','skingdoms','sphylums']]=tid_map[row.staxids]
    df.to_csv(f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split_edited/{x}",sep="\t")

In [70]:
import multiprocessing as mp
num_processes = 30  # Use all available CPU cores
pool = mp.Pool(processes=num_processes)
# Use the pool's map function to apply the get_data function to each element in td
result = pool.map(fill_missing_taxa, td)

In [31]:
import os
td=os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split")
td=[x for x in td if '.tsv' in x]
num_processes = 10  # Use all available CPU cores
pool = mp.Pool(processes=num_processes)

# Use the pool's map function to apply the get_data function to each element in td
result = pool.map(fill_missing_tax, td)

In [3]:
def get_data(x):
    df=pd.read_csv(f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split_edited/{x}",sep="\t",index_col=0, nrows=2000)
    ##exclude synthetic taxid
    df=df[df.staxids.astype(int)!=32630]
    ## exclude rhagoletis zephyria
    df=df[df.staxids.astype(int)!=28612]
    n_non_diptera,n_out_fam_diptera,AI,sseqid,slineage,sciname,tid,evalue,bitscore,qcovhsp,qstart,qend,pident=[np.nan]*13
    fam='Sciaridae;' 
    if df[~df.slineage.str.contains(fam)].shape[0]>0:
        top=df[~df.slineage.str.contains(fam)].iloc[0,:]
        sseqid,sciname,slineage,tid,evalue,bitscore,qcovhsp,qstart,qend,pident=top[['sseqid','sscinames','slineage','staxids','evalue','bitscore','qcovhsp','qstart','qend','pident']]
                   
        in_fam=df[df.slineage.str.contains(fam)&df.slineage.str.contains('Diptera')]
        out_fam_diptera=df[~df.slineage.str.contains(fam)&df.slineage.str.contains('Diptera')]

        ##return n_out_fam sseqid
        n_out_fam_diptera=len(set(out_fam_diptera['sseqid']))
        non_diptera=df[~df.slineage.str.contains('Diptera;')]

        ##return non dipteran sseqid
        n_non_diptera=len(set(non_diptera['sseqid']))
        AI=np.nan
        if out_fam_diptera.shape[0]>0 and non_diptera.shape[0]>0:
            AI=np.log10((out_fam_diptera.evalue.iloc[0]+1e-200)/(non_diptera.evalue.iloc[0]+1e-200))
        del in_fam, out_fam_diptera
            
    lo=[x.replace(".tsv",""),n_non_diptera,n_out_fam_diptera,AI,sseqid,slineage,sciname,tid,evalue,bitscore,qcovhsp,qstart,qend,pident]
    del df

    return lo



In [26]:

num_processes = 50 # Use all available CPU cores
pool = mp.Pool(processes=num_processes)
td=os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_results_split")
td=[x for x in td if '.tsv' in x]
# Use the pool's map function to apply the get_data function to each element in td
result = pool.map(get_data, td)
cols="accession,n_non_diptera,n_out_fam_diptera,AI,sseqid,slineage,sciname,tid,evalue,bitscore,qcovhsp,qstart,qend,pident".split(",")
dfresult=pd.DataFrame(result,columns=cols)


In [3]:
import pandas as pd
dfresult=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)


In [19]:

for index, row in dfresult.iterrows():
    non_dipt=(row.n_out_fam_diptera==0)&(row.evalue<1e-5)
    alien=row.AI>10
    if alien:
        dfresult.loc[index,'type']='alien'
    elif non_dipt:
        dfresult.loc[index,'type']='non_dipt'
    elif 'Diptera' in str(row.slineage):
        dfresult.loc[index,'type']='dipt'
    elif str(row.sseqid)=='nan':
        dfresult.loc[index,'type']='orphan'
    else:
        dfresult.loc[index,'type']='none'
    dfresult.loc[index,'HGT']=alien|non_dipt
    

In [28]:

input_file = open(f"ncbi_dataset/data/GCF_014529535.2/protein.faa") 
fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
htgs={x:fasta[x] for x in dfresult[dfresult['HGT']].accession}

SeqIO.write(
    htgs.values(),
    "outputs/DIAMOND_htgs.fasta",
    "fasta"
)

1721

In [1]:
%%bash
tar -czf outputs/diamond_clustered_results_split_edited.tar.gz -C "/n/netscratch/extavour_lab/Everyone/Rishabh/" "diamond_clustered_results_split_edited"

In [92]:
for index, row in dfresult.iterrows():
    dfresult.loc[index,'gene']=cds_gene[row.accession]
dfresult['length']=[len(fasta[x].seq) for x in dfresult.accession]
dfresult['description']=[" ".join(fasta[x].description.split(" ")[1:]).split("[Bradysia")[0] for x in dfresult.accession]


In [22]:
dfresult.to_csv("outputs/per_protein_diamond_data.tsv",sep="\t")

In [22]:

SeqIO.write(
    htgs.values(),
    "outputs/DIAMOND_htgs.fasta",
    "fasta"
)

1721

## add taxonomic information

In [16]:

def init_worker():
    global ncbi
    ncbi = NCBITaxa()
ncbi = NCBITaxa()


def get_ncbi_lineage_by_rank(taxid):
    
    """
    Return NCBI taxid and full lineage as {rank: scientific_name}
    for a taxid.
    """
    global ncbi
    try:
        lineage_taxids = ncbi.get_lineage(taxid)
        lineage_names = ncbi.get_taxid_translator(lineage_taxids)
        lineage_ranks = ncbi.get_rank(lineage_taxids)
    
        lineage_by_rank = {
            lineage_ranks[t]: lineage_names[t]
            for t in lineage_taxids
            if lineage_ranks.get(t) not in (None, "no rank")
        }
        return lineage_by_rank
    except:
        return 'None',{'None':'None'}
    

In [11]:
def get_sister_info(ranks):
    domain = kingdom = phylum = classt = order = None
    try:
        if 'Viruses' in str(ranks):
            domain='Viruses'
        else:
            domain=ranks['domain']
    except:
        superkingdom='None'
    try:
        kingdom=ranks['kingdom']
    except:
        kingdom='None'
    try:
        phylum=ranks['phylum']
    except:
        phylum='None'
    try:
        classt=ranks['class']
    except:
        classt='None'
    try:
        order=ranks['order']
    except:
        order='None'
    return [domain, kingdom, phylum, classt, order]


In [14]:
def get_data(tid):
    ranks=get_ncbi_lineage_by_rank(tid)
    return get_sister_info(ranks)
    

In [26]:
df=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
td=list(set([x for x in df.tid if str(x)!='nan']))
with mp.Pool(processes=50, initializer=init_worker) as pool:
    results = pool.map(get_data, td)
tmap={int(x):y for x,y in zip(td,results)}
for index, row in df.iterrows():
    if str(row.tid)!='nan':
        df.loc[index,["domain", "kingdom", "phylum", "class", "order"]]=tmap[int(row.tid)]


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 983966 was translated into 4903
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))


In [28]:
df.to_csv("outputs/per_protein_diamond_data.tsv",sep="\t")